<a href="https://colab.research.google.com/github/anteodor/Numerical-Methods/blob/main/lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratory Work: Numerical Methods

**Student:** Anastasiia Teodorovska  
**Language:** Python (Google Colab)

# Mathematical Background

This laboratory work focuses on numerical methods for solving systems of nonlinear and linear equations. The considered methods are based on iterative procedures, matrix factorization techniques, and elimination algorithms that are widely used in scientific computing and engineering applications.

## Newton's Method for Nonlinear Systems

Newton's method is an iterative technique used to solve systems of nonlinear equations of the form

$$
F(x)=0.
$$

Starting from an initial approximation $x^{(0)}$, the solution is refined according to

$$
x^{(k+1)} = x^{(k)} - J^{-1}(x^{(k)})F(x^{(k)}),
$$

where $J(x)$ is the Jacobian matrix of the system.

In practice, the correction vector is obtained by solving the linear system

$$
J(x^{(k)})\,\Delta x^{(k)}
=
-F(x^{(k)}).
$$

followed by the update

$$
x^{(k+1)}
=
x^{(k)}
+
\Delta x^{(k)}.
$$

Newton's method is known for its quadratic convergence near the solution, making it one of the most efficient methods for solving nonlinear systems.

---

## Aitken Acceleration

Aitken's $\Delta^2$ - process is a technique used to accelerate the convergence of an iterative sequence.

For three consecutive approximations

$$
x_k,\quad x_{k+1},\quad x_{k+2},
$$

the accelerated approximation is computed as

$$
x_k^{(A)} =
x_k -
\frac{(x_{k+1}-x_k)^2}
{x_{k+2}-2x_{k+1}+x_k}.
$$

The method is particularly useful when the original iterative process converges slowly. In this laboratory work, Aitken acceleration is applied to the sequence generated by Newton's method.

---

## Gaussian Elimination

Gaussian elimination is a direct method for solving a system of linear equations

$$
Ax=b.
$$

The main idea is to transform the coefficient matrix into an upper triangular form using elementary row operations.

After elimination, the system takes the form

$$
Ux=c,
$$

where $U$ is an upper triangular matrix.

The solution is then obtained using backward substitution, starting from the last equation and proceeding upward.

---

## Partial Pivoting

Partial pivoting is an enhancement of Gaussian elimination designed to improve numerical stability.

At each elimination step, the row containing the largest absolute value in the current pivot column is selected as the pivot row.

This operation reduces the effect of round-off errors and avoids divisions by very small numbers, which may otherwise lead to significant numerical inaccuracies.

---

## LU Decomposition

LU decomposition is a matrix factorization technique that represents a matrix as the product of a lower triangular matrix and an upper triangular matrix.

When pivoting is included, the decomposition is written as

$$
PA = LU,
$$

where

- $P$ is a permutation matrix;
- $L$ is a lower triangular matrix;
- $U$ is an upper triangular matrix.

The original system

$$
Ax=b
$$

is transformed into

$$
LUx = Pb.
$$

The solution is obtained in two stages:

1. Forward substitution:

$$
Ly = Pb,
$$

2. Backward substitution:

$$
Ux = y.
$$

LU decomposition is particularly efficient when multiple systems with the same coefficient matrix but different right-hand side vectors must be solved.


## 1. Newton Method and Aitken Acceleration

The objective of this task is to investigate whether Aitken’s acceleration can improve the convergence rate of Newton’s method for a slowly convergent nonlinear system.

The considered nonlinear system is

$$
\begin{cases}
x^2 + 3y^3 = 0, \\
x + y = 0.
\end{cases}
$$

Newton’s method is a widely used iterative technique for solving nonlinear systems. However, some problems may exhibit slow convergence, requiring a relatively large number of iterations to reach the desired accuracy. To accelerate the convergence process, Aitken’s acceleration technique can be applied to the sequence of approximations generated by Newton’s method.

In this experiment, two approaches are compared:

- **Classical Newton’s method**
- **Newton’s method with Aitken’s acceleration**

For both methods, the same initial approximation is used:

$$
x^{(0)} = (2, 1).
$$

The performance of both approaches is evaluated by comparing the number of iterations required to achieve convergence and by analyzing the residual vector \(F(x)\) at the computed solution.

In [6]:
import numpy as np
import pandas as pd

# Nonlinear system with slow convergence
def function_slow(xy):
    x, y = xy
    return np.array([
        x**2 + 3*y**3,
        x + y
    ], dtype=float)


# Jacobian matrix of the nonlinear system
def jacobian_slow(xy):
    x, y = xy
    return np.array([
        [2*x, 9*y**2],
        [1, 1]
    ], dtype=float)


def newton_method(fun, jacobian, x_init, epsilon=1e-8, max_iter=150):
    """
    Classical Newton method for solving a nonlinear system F(x) = 0.
    """
    x = np.array(x_init, dtype=float)
    history = []

    for iteration in range(1, max_iter + 1):
        F = fun(x)
        J = jacobian(x)

        # Solve the linear system J(x_k) * delta = -F(x_k)
        delta = np.linalg.solve(J, -F)
        x_new = x + delta

        step_norm = np.linalg.norm(delta)
        residual_norm = np.linalg.norm(fun(x_new))

        history.append({
            "Iteration": iteration,
            "x": x_new[0],
            "y": x_new[1],
            "Step norm": step_norm,
            "Residual norm": residual_norm
        })

        x = x_new

        if step_norm < epsilon or residual_norm < epsilon:
            break

    return x, history


def aitken_accelerated_newton(fun, jacobian, x_init, epsilon=1e-8, max_iter=50):
    """
    Newton method accelerated by Aitken's process.

    Three consecutive Newton approximations are generated, and then
    Aitken acceleration is applied component-wise to improve convergence.
    """
    x = np.array(x_init, dtype=float)
    history = []
    newton_steps = 0

    for iteration in range(1, max_iter + 1):
        approximations = [x]

        # Generate two consecutive Newton approximations:
        # x_k, x_{k+1}, x_{k+2}
        for _ in range(2):
            F = fun(approximations[-1])
            J = jacobian(approximations[-1])

            delta = np.linalg.solve(J, -F)
            x_next = approximations[-1] + delta
            approximations.append(x_next)
            newton_steps += 1

        x0, x1, x2 = approximations

        # Component-wise Aitken acceleration:
        # x_acc = x0 - (x1 - x0)^2 / (x2 - 2*x1 + x0)
        denominator = x2 - 2*x1 + x0

        if np.all(np.abs(denominator) > 1e-14):
            x_acc = x0 - ((x1 - x0) ** 2) / denominator
        else:
            # If the denominator is too small, use the last Newton approximation
            x_acc = x2

        step_norm = np.linalg.norm(x_acc - x)
        residual_norm = np.linalg.norm(fun(x_acc))

        history.append({
            "Iteration": iteration,
            "Newton steps": newton_steps,
            "x": x_acc[0],
            "y": x_acc[1],
            "Step norm": step_norm,
            "Residual norm": residual_norm
        })

        x = x_acc

        if step_norm < epsilon or residual_norm < epsilon:
            break

    return x, history


# Initial approximation
x0 = [2.0, 1.0]

# Solve the system using the classical Newton method
solution_newton, history_newton = newton_method(
    function_slow,
    jacobian_slow,
    x0
)

# Solve the system using Newton's method with Aitken acceleration
solution_aitken, history_aitken = aitken_accelerated_newton(
    function_slow,
    jacobian_slow,
    x0
)

# Create a comparison table
comparison = pd.DataFrame({
    "Method": [
        "Classical Newton method",
        "Newton method with Aitken acceleration"
    ],
    "Solution x": [
        solution_newton[0],
        solution_aitken[0]
    ],
    "Solution y": [
        solution_newton[1],
        solution_aitken[1]
    ],
    "Iterations": [
        len(history_newton),
        len(history_aitken)
    ],
    "Residual norm ||F(x)||": [
        np.linalg.norm(function_slow(solution_newton)),
        np.linalg.norm(function_slow(solution_aitken))
    ],
    "F(solution)": [
        function_slow(solution_newton),
        function_slow(solution_aitken)
    ]
})

comparison

,Method,Solution x,Solution y,Iterations,Residual norm ||F(x)||,F(solution)
0,Classical Newton method,-0.000072,0.000072,18,5.201175e-09,"[5.201175368944841e-09, 0.0]"
1,Newton method with Aitken acceleration,0.333333,-0.333333,10,2.775558e-17,"[2.7755575615628914e-17, 0.0]"


In [7]:
# Iteration history for the classical Newton method
newton_history_df = pd.DataFrame(history_newton)
newton_history_df

,Iteration,x,y,Step norm,Residual norm
0,1,-2.000000,2.000000,4.123106,2.800000e+01
1,2,-1.300000,1.300000,0.989949,8.281000e+00
2,3,-0.835036,0.835036,0.657558,2.444064e+00
3,4,-0.527439,0.527439,0.435009,7.183785e-01
4,5,-0.325568,0.325568,0.285489,2.095192e-01
5,6,-0.195033,0.195033,0.184604,6.029371e-02
6,7,-0.112710,0.112710,0.116422,1.699903e-02
7,8,-0.062677,0.062677,0.070758,4.666992e-03
8,9,-0.033636,0.033636,0.041069,1.245573e-03
9,10,-0.017555,0.017555,0.022742,3.244149e-04


In [8]:
# Iteration history for Newton method with Aitken acceleration
aitken_history_df = pd.DataFrame(history_aitken)
aitken_history_df

,Iteration,Newton steps,x,y,Step norm,Residual norm
0,1,2,-1.404255,1.588235,3.454703e+00,1.399207e+01
1,2,4,8.844702,-0.041629,1.037774e+01,7.872228e+01
2,3,6,2.297914,-3.344269,7.332658e+00,1.069330e+02
3,4,8,2.298311,-0.056153,3.288116e+00,5.737912e+00
4,5,10,0.672973,-0.896634,1.829790e+00,1.724230e+00
5,6,12,0.674602,-0.143604,7.530319e-01,6.935817e-01
6,7,14,0.336623,-0.345673,3.937780e-01,1.393582e-02
7,8,16,0.332095,-0.333214,1.325578e-02,1.322591e-03
8,9,18,0.333333,-0.333334,1.243732e-03,2.353095e-07
9,10,20,0.333333,-0.333333,1.806045e-07,2.775558e-17


### Analysis of the Results

The obtained results show that both methods successfully converged to valid solutions of the nonlinear system. This is confirmed by the residual norms, which are close to zero in both cases.

The classical Newton method converged to

$$
(x,y)\approx(-0.000072,\;0.000072),
$$

which is numerically equivalent to the root

$$
(x,y)=(0,0).
$$

The method required 18 iterations and produced a residual norm of

$$
\|F(x)\|\approx 5.20\times10^{-9}.
$$

The Newton method with Aitken acceleration converged to another valid root,

$$
(x,y)\approx(0.333333,\;-0.333333),
$$

which corresponds to

$$
(x,y)=\left(\frac{1}{3},-\frac{1}{3}\right).
$$

The method reached convergence after 10 accelerated iterations and achieved a significantly smaller residual norm,

$$
\|F(x)\|\approx 2.78\times10^{-17}.
$$

The two methods converged to different roots of the nonlinear system. This behavior is expected because the system possesses multiple solutions, and the convergence path depends on the sequence of approximations generated during the iterative process.

Although both approaches achieved highly accurate solutions, the Aitken-accelerated version reached a substantially smaller residual norm while requiring fewer outer iterations. The results demonstrate that Aitken's acceleration can effectively improve the convergence behavior of Newton's method for slowly convergent nonlinear systems.

## 2. Manual Solution Using Gaussian Elimination

In this task, two systems of linear equations are solved manually using Gaussian elimination.  
The main idea of the method is to eliminate unknowns step by step, transform the system into an upper triangular form, and then apply back substitution.

---

### First system

The first system is

$$
\begin{cases}
4x_1 - x_2 + x_3 = 8, \\
2x_1 + 5x_2 + 2x_3 = 3, \\
x_1 + 2x_2 + 4x_3 = 11.
\end{cases}
$$

After applying Gaussian elimination, the reduced system for $x_2$ and $x_3$ becomes

$$
\begin{cases}
-11x_2 - 3x_3 = 2, \\
2x_2 + 5x_3 = 13.
\end{cases}
$$

To eliminate $x_2$, multiply the second equation by $5.5$:

$$
5.5(2x_2 + 5x_3 = 13)
$$

which gives

$$
11x_2 + 27.5x_3 = 71.5.
$$

Adding this equation to the first reduced equation gives

$$
(-11x_2 - 3x_3) + (11x_2 + 27.5x_3) = 2 + 71.5.
$$

Therefore,

$$
24.5x_3 = 73.5,
$$

and hence

$$
x_3 = 3.
$$

Substituting $x_3 = 3$ into

$$
2x_2 + 5x_3 = 13
$$

gives

$$
2x_2 + 15 = 13,
$$

so

$$
x_2 = -1.
$$

Finally, substituting $x_2 = -1$ and $x_3 = 3$ into the first original equation,

$$
4x_1 - x_2 + x_3 = 8,
$$

we obtain

$$
4x_1 - (-1) + 3 = 8,
$$

$$
4x_1 = 4,
$$

and therefore

$$
x_1 = 1.
$$

Thus, the solution of the first system is

$$
\boxed{x_1 = 1,\quad x_2 = -1,\quad x_3 = 3}.
$$

---

### Second system

The second printed system is

$$
\begin{cases}
4x_1 - x_2 + 2x_3 = 9, \\
2x_1 + 4x_2 - x_3 = -5, \\
x_1 + x_2 - 3x_3 = -9.
\end{cases}
$$

We eliminate $x_1$ from the second equation using

$$
EQ_1 - 2EQ_2.
$$

This gives

$$
(4x_1 - x_2 + 2x_3) - 2(2x_1 + 4x_2 - x_3) = 9 - 2(-5),
$$

so

$$
-9x_2 + 4x_3 = 19.
$$

Next, we eliminate $x_1$ from the third equation using

$$
EQ_1 - 4EQ_3.
$$

This gives

$$
(4x_1 - x_2 + 2x_3) - 4(x_1 + x_2 - 3x_3) = 9 - 4(-9),
$$

so

$$
-5x_2 + 14x_3 = 45.
$$

Therefore, the reduced system is

$$
\begin{cases}
-9x_2 + 4x_3 = 19, \\
-5x_2 + 14x_3 = 45.
\end{cases}
$$

To eliminate $x_2$, multiply the first equation by $5$ and the second equation by $9$:

$$
-45x_2 + 20x_3 = 95,
$$

$$
-45x_2 + 126x_3 = 405.
$$

Subtracting the first equation from the second gives

$$
106x_3 = 310.
$$

Hence,

$$
x_3 = \frac{310}{106} = \frac{155}{53}.
$$

Now substitute $x_3 = \frac{155}{53}$ into

$$
-9x_2 + 4x_3 = 19.
$$

Then

$$
-9x_2 + 4 \cdot \frac{155}{53} = 19,
$$

$$
-9x_2 + \frac{620}{53} = \frac{1007}{53},
$$

$$
-9x_2 = \frac{387}{53},
$$

and therefore

$$
x_2 = -\frac{43}{53}.
$$

Finally, substitute $x_2 = -\frac{43}{53}$ and $x_3 = \frac{155}{53}$ into the third original equation:

$$
x_1 + x_2 - 3x_3 = -9.
$$

This gives

$$
x_1 - \frac{43}{53} - 3\cdot\frac{155}{53} = -9.
$$

Thus,

$$
x_1 - \frac{508}{53} = -\frac{477}{53},
$$

so

$$
x_1 = \frac{31}{53}.
$$

Therefore, for the printed second system, the solution is

$$
\boxed{
x_1 = \frac{31}{53},\quad
x_2 = -\frac{43}{53},\quad
x_3 = \frac{155}{53}
}.
$$

In decimal form,

$$
\boxed{
x_1 \approx 0.5849,\quad
x_2 \approx -0.8113,\quad
x_3 \approx 2.9245
}.
$$

---

### Verification of the Stated Solution

The task states that both systems have the solution

$$
x_1 = 1,\quad x_2 = -1,\quad x_3 = 3.
$$

For the first system, this solution is correct. However, for the second printed system, direct substitution gives

$$
4(1) - (-1) + 2(3) = 11 \neq 9.
$$

Therefore, the stated solution does not satisfy the printed second system. This indicates that there is most likely a typo or mismatch in the second system as written in the task.

---

### Conclusion

The first system was successfully solved manually using Gaussian elimination, and the obtained solution is

$$
\boxed{x_1 = 1,\quad x_2 = -1,\quad x_3 = 3}.
$$

For the second printed system, Gaussian elimination gives

$$
\boxed{
x_1 = \frac{31}{53},\quad
x_2 = -\frac{43}{53},\quad
x_3 = \frac{155}{53}
}.
$$

This result shows that the printed second system is inconsistent with the stated solution from the task.

## 3. Gaussian Elimination with Partial Pivoting

In this task, Gaussian elimination with partial pivoting is implemented in Python to solve a system of linear equations of the form

$$
Ax = b.
$$

The main purpose of partial pivoting is to improve the numerical stability of the Gaussian elimination process. At each elimination step, the algorithm selects the row with the largest absolute value in the current pivot column and swaps it with the current row. This reduces the risk of division by a very small pivot element and helps avoid unnecessary numerical errors.

The implemented algorithm consists of the following stages:

- creating copies of the input matrix \(A\) and vector \(b\), so that the original data are not modified;
- forming the augmented matrix \([A|b]\);
- applying partial pivoting at each elimination step;
- transforming the system into an upper triangular form;
- applying back substitution to compute the solution vector \(x\);
- comparing the obtained result with the solution returned by `scipy.linalg.solve`.

The comparison with the SciPy solver is used to verify the correctness of the implemented method. Additionally, the residual vector

$$
r = Ax - b
$$

and its norm

$$
\|r\|
$$

are computed to evaluate the accuracy of the numerical solution.

In [11]:
import numpy as np
import scipy.linalg


def linearsolver_pivoting(A, b, tolerance=1e-12):
    """
    Solves a system of linear equations Ax = b
    using Gaussian elimination with partial pivoting.
    """

    # Convert input data to NumPy arrays and make independent copies
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)

    n = len(b)

    # Create the augmented matrix [A | b]
    M = np.hstack((A, b.reshape(-1, 1)))

    # Forward elimination with partial pivoting
    for k in range(n - 1):

        # Select the pivot row with the largest absolute value in column k
        pivot_row = np.argmax(np.abs(M[k:, k])) + k

        # Check whether the pivot element is too close to zero
        if abs(M[pivot_row, k]) < tolerance:
            raise ValueError("The matrix is singular or nearly singular.")

        # Swap the current row with the pivot row if needed
        if pivot_row != k:
            M[[k, pivot_row]] = M[[pivot_row, k]]

        # Eliminate all elements below the pivot
        for i in range(k + 1, n):
            factor = M[i, k] / M[k, k]
            M[i, k:] -= factor * M[k, k:]

    # Check the last pivot before back substitution
    if abs(M[n - 1, n - 1]) < tolerance:
        raise ValueError("The matrix is singular or nearly singular.")

    # Back substitution
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        known_sum = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, -1] - known_sum) / M[i, i]

    return x, M


def solve_and_verify(A, b, system_name):
    """
    Solves a linear system using the implemented Gaussian elimination
    with partial pivoting and compares the result with scipy.linalg.solve.
    """

    # Convert inputs to NumPy arrays
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    # Save original matrix and vector to verify that they are not modified
    A_original = A.copy()
    b_original = b.copy()

    # Solve the system using the implemented method
    x_custom, upper_augmented_matrix = linearsolver_pivoting(A, b)

    # Solve the same system using SciPy for comparison
    x_scipy = scipy.linalg.solve(A, b)

    # Compute verification values
    difference = np.abs(x_custom - x_scipy)
    residual = A @ x_custom - b
    residual_norm = np.linalg.norm(residual)

    print("=" * 80)
    print(system_name)
    print("=" * 80)

    print("\nOriginal matrix A:")
    print(A_original)

    print("\nOriginal vector b:")
    print(b_original)

    print("\nUpper triangular augmented matrix after elimination:")
    print(upper_augmented_matrix)

    print("\nSolution using Gaussian elimination with partial pivoting:")
    print(x_custom)

    print("\nSolution using scipy.linalg.solve:")
    print(x_scipy)

    print("\nAbsolute difference between the two solutions:")
    print(difference)

    print("\nResidual vector Ax - b:")
    print(residual)

    print("\nResidual norm ||Ax - b||:")
    print(residual_norm)

    print("\nWas the original matrix A modified?")
    print(not np.array_equal(A, A_original))

    print("\nWas the original vector b modified?")
    print(not np.array_equal(b, b_original))

    print("\n")


# System from the assignment

A_task = np.array([
    [4, -1, 1],
    [2, 5, 2],
    [1, 2, 4]
], dtype=float)

b_task = np.array([8, 3, 11], dtype=float)

solve_and_verify(
    A_task,
    b_task,
    "System from the assignment"
)


# Additional 4x4 system for extended verification

A_extra = np.array([
    [2, 1, 1, 0],
    [4, 3, 3, 1],
    [8, 7, 9, 5],
    [6, 7, 9, 8]
], dtype=float)

b_extra = np.array([1, 2, 3, 4], dtype=float)

solve_and_verify(
    A_extra,
    b_extra,
    "Additional 4x4 system for extended verification"
)

System from the assignment

Original matrix A:
[[ 4. -1.  1.]
 [ 2.  5.  2.]
 [ 1.  2.  4.]]

Original vector b:
[ 8.  3. 11.]

Upper triangular augmented matrix after elimination:
[[ 4.         -1.          1.          8.        ]
 [ 0.          5.5         1.5        -1.        ]
 [ 0.          0.          3.13636364  9.40909091]]

Solution using Gaussian elimination with partial pivoting:
[ 1. -1.  3.]

Solution using scipy.linalg.solve:
[ 1. -1.  3.]

Absolute difference between the two solutions:
[0. 0. 0.]

Residual vector Ax - b:
[0. 0. 0.]

Residual norm ||Ax - b||:
0.0

Was the original matrix A modified?
False

Was the original vector b modified?
False


Additional 4x4 system for extended verification

Original matrix A:
[[2. 1. 1. 0.]
 [4. 3. 3. 1.]
 [8. 7. 9. 5.]
 [6. 7. 9. 8.]]

Original vector b:
[1. 2. 3. 4.]

Upper triangular augmented matrix after elimination:
[[ 8.          7.          9.          5.          3.        ]
 [ 0.          1.75        2.25        4.25    

### Analysis of the Results

The implemented Gaussian elimination algorithm with partial pivoting successfully solved the system of linear equations provided in the laboratory assignment.

During forward elimination, partial pivoting was applied by selecting the row with the largest absolute value in the current pivot column. As a result, the augmented matrix was transformed into an upper triangular form, which allowed the solution to be obtained by back substitution.

The computed solution is

$$
x = \left(1,-1,3\right).
$$

This result is identical to the solution obtained using the library function `scipy.linalg.solve`:

$$
x_{\text{SciPy}} = \left(1,-1,3\right).
$$

The absolute difference between the two solutions is

$$
|x_{\text{custom}} - x_{\text{SciPy}}|
=
\left(0,0,0\right),
$$

which confirms that the custom implementation produces the same result as the reference solver.

The residual vector is

$$
Ax - b =
\left(0,0,0\right),
$$

and the residual norm is

$$
\|Ax-b\| = 0.
$$

Since this value is equal to zero, the computed solution satisfies the original system exactly within machine precision.

The output also confirms that the original matrix $A$ and vector $b$ were not modified during the computation, because independent copies of the input data were used inside the solver.

To further verify the correctness and robustness of the implementation, an additional 4×4 system of linear equations was solved.

The computed solution is

$$
x = \left(1,0.5,-1.5,1\right).
$$

This result is identical to the solution obtained using the library function `scipy.linalg.solve`:

$$
x_{\text{SciPy}} = \left(1,0.5,-1.5,1\right).
$$

The absolute difference between the two solutions is

$$
|x_{\text{custom}} - x_{\text{SciPy}}|
=
\left(0,0,0,0\right),
$$

which confirms that the custom implementation produces the same result as the reference solver.

The residual vector is

$$
Ax - b =
\left(
-3.33 \times 10^{-16},
0,
0,
1.78 \times 10^{-15}
\right),
$$

and the residual norm is

$$
\|Ax-b\|
\approx
1.81 \times 10^{-15}.
$$

Since this value is extremely close to zero, the computed solution satisfies the original system with high numerical accuracy.

The output also confirms that the original matrix $A$ and vector $b$ were not modified during the computation, because independent copies of the input data were used inside the solver.

Therefore, the developed Gaussian elimination solver with partial pivoting was verified to be correct, accurate, and numerically reliable for both tested systems.

## 4. LU Decomposition with Partial Pivoting

In this task, the LU decomposition algorithm with partial pivoting is extended in order to solve a system of linear equations of the form

$$
Ax = b.
$$

LU decomposition represents the coefficient matrix $A$ as a product of two triangular matrices. When pivoting is used, row permutations are also taken into account. Therefore, the decomposition is written as

$$
PA = LU,
$$

where:

- $P$ is the permutation matrix, which represents row swaps performed during pivoting;
- $L$ is the lower triangular matrix with ones on the main diagonal;
- $U$ is the upper triangular matrix.

Partial pivoting is used to improve numerical stability. At each step, the row with the largest absolute value in the current pivot column is selected as the pivot row. This reduces the risk of division by a very small pivot element and makes the decomposition more reliable.

After the decomposition is obtained, the original system

$$
Ax = b
$$

is transformed using the relation

$$
PA = LU.
$$

Multiplying both sides of the system by $P$, we get

$$
PAx = Pb.
$$

Since $PA = LU$, the system becomes

$$
LUx = Pb.
$$

This system is solved in two triangular stages. First, forward substitution is used to solve

$$
Ly = Pb.
$$

Then, backward substitution is applied to solve

$$
Ux = y.
$$

The obtained solution is finally compared with the result returned by `scipy.linalg.solve`. In addition, the residual norm

$$
\|Ax-b\|
$$

is computed to verify the numerical accuracy of the solution.

In [10]:
import numpy as np
import scipy.linalg


def lu_decomposition_partial_pivoting(A, tolerance=1e-12):
    """
    Performs LU decomposition with partial pivoting.

    The decomposition is computed in the form:
        P @ A = L @ U

    """

    A = np.array(A, dtype=float, copy=True)

    if A.shape[0] != A.shape[1]:
        raise ValueError("Matrix A must be square.")

    n = A.shape[0]

    P = np.eye(n)
    L = np.zeros((n, n))
    U = A.copy()

    for k in range(n - 1):

        # Select pivot row using partial pivoting
        pivot_row = np.argmax(np.abs(U[k:, k])) + k

        # Check whether the pivot is zero or too close to zero
        if abs(U[pivot_row, k]) < tolerance:
            raise ValueError("Matrix is singular or nearly singular.")

        # Swap rows in U and P if necessary
        if pivot_row != k:
            U[[k, pivot_row], :] = U[[pivot_row, k], :]
            P[[k, pivot_row], :] = P[[pivot_row, k], :]

            # Swap only the previously computed part of L
            if k > 0:
                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]

        # Eliminate entries below the pivot
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    # Check the last pivot
    if abs(U[n - 1, n - 1]) < tolerance:
        raise ValueError("Matrix is singular or nearly singular.")

    # The diagonal elements of L are equal to 1
    np.fill_diagonal(L, 1.0)

    return P, L, U


def forward_substitution(L, b, tolerance=1e-12):
    """
    Solves a lower triangular system Ly = b using forward substitution.
    """

    L = np.array(L, dtype=float)
    b = np.array(b, dtype=float)

    n = len(b)
    y = np.zeros(n)

    for i in range(n):
        if abs(L[i, i]) < tolerance:
            raise ValueError("Zero diagonal element detected in L.")

        known_sum = np.dot(L[i, :i], y[:i])
        y[i] = (b[i] - known_sum) / L[i, i]

    return y


def backward_substitution(U, y, tolerance=1e-12):
    """
    Solves an upper triangular system Ux = y using backward substitution.
    """

    U = np.array(U, dtype=float)
    y = np.array(y, dtype=float)

    n = len(y)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        if abs(U[i, i]) < tolerance:
            raise ValueError("Zero diagonal element detected in U.")

        known_sum = np.dot(U[i, i + 1:], x[i + 1:])
        x[i] = (y[i] - known_sum) / U[i, i]

    return x


def solve_lu_pivoting(A, b):
    """
    Solves a linear system Ax = b using LU decomposition with partial pivoting.
    """

    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)

    if A.shape[0] != A.shape[1]:
        raise ValueError("Matrix A must be square.")

    if A.shape[0] != b.shape[0]:
        raise ValueError("The dimensions of A and b are incompatible.")

    P, L, U = lu_decomposition_partial_pivoting(A)

    # Since P @ A = L @ U, we solve:
    # L @ U @ x = P @ b
    Pb = P @ b

    # First solve Ly = Pb
    y = forward_substitution(L, Pb)

    # Then solve Ux = y
    x = backward_substitution(U, y)

    return x, P, L, U, Pb, y


# System from the lecture
A = np.array([
    [1,  2,  1],
    [1, -2,  2],
    [2, 12, -2]
], dtype=float)

b = np.array([0, 4, 4], dtype=float)

# Solve using the custom LU decomposition with partial pivoting
x_custom, P, L, U, Pb, y = solve_lu_pivoting(A, b)

# Solve using SciPy for verification
x_scipy = scipy.linalg.solve(A, b)

# Compute verification values
PA = P @ A
LU = L @ U
difference_pa_lu = PA - LU
difference_solutions = np.abs(x_custom - x_scipy)
residual = A @ x_custom - b
residual_norm = np.linalg.norm(residual)

print("Matrix A:")
print(A)

print("\nVector b:")
print(b)

print("\nPermutation matrix P:")
print(P)

print("\nLower triangular matrix L:")
print(L)

print("\nUpper triangular matrix U:")
print(U)

print("\nCheck PA = LU:")
print("P @ A =")
print(PA)

print("\nL @ U =")
print(LU)

print("\nDifference PA - LU:")
print(difference_pa_lu)

print("\nRight-hand side after pivoting Pb:")
print(Pb)

print("\nIntermediate solution y from Ly = Pb:")
print(y)

print("\nSolution using custom LU decomposition with partial pivoting:")
print(x_custom)

print("\nSolution using scipy.linalg.solve:")
print(x_scipy)

print("\nAbsolute difference between the two solutions:")
print(difference_solutions)

print("\nResidual vector Ax - b:")
print(residual)

print("\nResidual norm ||Ax - b||:")
print(residual_norm)

Matrix A:
[[ 1.  2.  1.]
 [ 1. -2.  2.]
 [ 2. 12. -2.]]

Vector b:
[0. 4. 4.]

Permutation matrix P:
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]

Lower triangular matrix L:
[[1.  0.  0. ]
 [0.5 1.  0. ]
 [0.5 0.5 1. ]]

Upper triangular matrix U:
[[ 2.  12.  -2. ]
 [ 0.  -8.   3. ]
 [ 0.   0.   0.5]]

Check PA = LU:
P @ A =
[[ 2. 12. -2.]
 [ 1. -2.  2.]
 [ 1.  2.  1.]]

L @ U =
[[ 2. 12. -2.]
 [ 1. -2.  2.]
 [ 1.  2.  1.]]

Difference PA - LU:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Right-hand side after pivoting Pb:
[4. 4. 0.]

Intermediate solution y from Ly = Pb:
[ 4.  2. -3.]

Solution using custom LU decomposition with partial pivoting:
[11.  -2.5 -6. ]

Solution using scipy.linalg.solve:
[11.  -2.5 -6. ]

Absolute difference between the two solutions:
[0. 0. 0.]

Residual vector Ax - b:
[0. 0. 0.]

Residual norm ||Ax - b||:
0.0


### Analysis of the Results

The system of linear equations was successfully solved using LU decomposition with partial pivoting.

The decomposition was performed in the form

$$
PA = LU.
$$

The correctness of the decomposition was verified by comparing the matrices $PA$ and $LU$. Since their difference is the zero matrix,

$$
PA - LU =
\begin{pmatrix}
0 & 0 & 0 \\
0 & 0 & 0 \\
0 & 0 & 0
\end{pmatrix},
$$

the obtained decomposition is correct.

After applying the permutation matrix to the right-hand side vector, we obtained

$$
Pb = (4,\;4,\;0).
$$

Then the triangular system

$$
Ly = Pb
$$

was solved using forward substitution, giving

$$
y = (4,\;2,\;-3).
$$

Next, backward substitution was applied to solve

$$
Ux = y.
$$

The final solution is

$$
x = (11,\;-2.5,\;-6).
$$

The obtained solution was compared with the result returned by `scipy.linalg.solve`:

$$
x_{\text{SciPy}} = (11,\;-2.5,\;-6).
$$

The absolute difference between the two solutions is

$$
|x_{\text{custom}} - x_{\text{SciPy}}| = (0,\;0,\;0).
$$

The residual vector is

$$
Ax-b = (0,\;0,\;0),
$$

and the residual norm is

$$
\|Ax-b\| = 0.
$$

This confirms that the computed solution satisfies the original system exactly within numerical precision.

Therefore, the implemented LU decomposition with partial pivoting, together with forward and backward substitution, was verified to be correct, accurate, and numerically reliable for the given system.

## Conclusions

In this laboratory work, several numerical methods for solving systems of nonlinear and linear equations were studied and implemented. The obtained results confirmed both the correctness of the implemented algorithms and the practical importance of numerical techniques for solving mathematical problems.

First, Newton's method and Newton's method accelerated by Aitken's process were applied to a slowly convergent nonlinear system. Both approaches successfully converged to valid roots of the system and produced very small residual norms. The results demonstrated that Aitken's acceleration can significantly improve the convergence behavior of Newton's method and reduce the number of outer iterations required to achieve a highly accurate solution.

Next, Gaussian elimination was applied manually to two systems of linear equations. The first system was solved successfully, yielding the expected solution. During the analysis of the second system, it was discovered that the solution stated in the assignment did not satisfy the printed equations. By performing Gaussian elimination and verifying the results through substitution, the correct solution of the printed system was obtained. This highlights the importance of validating numerical results rather than relying solely on the provided answers.

The Gaussian elimination algorithm with partial pivoting was then implemented in Python. Partial pivoting improved numerical stability by selecting the largest available pivot element at each elimination step. The computed solution was identical to the result returned by `scipy.linalg.solve`, while the residual norm was close to machine precision, confirming the correctness and accuracy of the implementation.

Finally, LU decomposition with partial pivoting was implemented and used to solve a system of linear equations through forward and backward substitution. The decomposition was verified by confirming the equality $PA = LU$, and the resulting solution matched exactly the one obtained using SciPy. The residual norm was equal to zero, demonstrating the reliability of the implemented algorithm.

Overall, the laboratory work provided practical experience with iterative methods, Gaussian elimination, pivoting strategies, and LU decomposition. The obtained results confirmed that these numerical methods are accurate, efficient, and suitable for solving systems of equations encountered in scientific and engineering computations.
